In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from imblearn.over_sampling import SMOTE

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, precision_recall_curve, auc, roc_auc_score
)

# ==============================================================================
# 1. 실제 데이터 로드 및 Target 정제
# ==============================================================================
file_path = 'secom_25%_중앙값대치_이상치처리.xlsx'
df = pd.read_excel(file_path)

y = df['Pass_Fail'].map({-1: 0, 1: 1})
X = df.drop(columns=['Result', 'Pass_Fail'])

# ==============================================================================
# [3주차 Step 1] Feature Selection 및 데이터 고도화 함수 정의
# ==============================================================================
def apply_feature_selection(X_train, y_train, X_val, n_features=30):
    """
    Data Leakage 방지를 위해 Train Fold 데이터로만 단계별 Feature Selection 수행
    1단계: Correlation Filtering (상관관계 0.95 이상 제거)
    2단계: ANOVA F-test / Mutual Info 기반 상위 변수 선택
    3단계: RFE (Logistic Regression 기반) final 선택
    """
    # 1. Correlation Filtering (Train 기준)
    corr_matrix = X_train.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

    X_train_filter = X_train.drop(columns=to_drop)
    X_val_filter = X_val.drop(columns=to_drop)

    # 2. ANOVA F-test 기반 상위 후보 추출 (100개)
    k_anova = min(100, X_train_filter.shape[1])
    selector_f = SelectKBest(score_func=f_classif, k=k_anova)
    selector_f.fit(X_train_filter, y_train)

    cols_anova = X_train_filter.columns[selector_f.get_support()]
    X_train_f = X_train_filter[cols_anova]
    X_val_f = X_val_filter[cols_anova]

    # 3. RFE 기반 최종 n_features 추출
    rfe_estimator = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    k_rfe = min(n_features, X_train_f.shape[1])
    rfe = RFE(estimator=rfe_estimator, n_features_to_select=k_rfe, step=0.1)
    rfe.fit(X_train_f, y_train)

    selected_cols = X_train_f.columns[rfe.get_support()]

    return (
        X_train_f[selected_cols],
        X_val_f[selected_cols],
        list(selected_cols)
    )

# ==============================================================================
# 2. 모델 딕셔너리 구성
# ==============================================================================
scale_pos_weight = (len(y) - sum(y)) / sum(y)

models = {
    'kNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42)
}

# ==============================================================================
# 3. [3주차 Step 2] SMOTE + Feature Selection + Threshold Tuning 교차 검증
# ==============================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
comparison_results = []
pr_curves_data = {}
feature_importances_dict = {}

# 최적화 분류 임계값 설정 (기존 0.5 -> 0.3으로 하향 조정하여 미탐/FN 최소화)
CUSTOM_THRESHOLD = 0.3

for model_name, model in models.items():
    metrics = {
        'Accuracy': [], 'Precision': [], 'Recall': [],
        'F1': [], 'PR-AUC': [], 'ROC-AUC': []
    }

    all_y_true = []
    all_y_proba = []
    selected_features_list = []

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # 1. 결측치 대치 및 스케일링 (Train fit)
        imputer = SimpleImputer(strategy='median')
        scaler = StandardScaler()

        X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
        X_val_imp = pd.DataFrame(imputer.transform(X_val), columns=X.columns)

        # 2. 단계별 Feature Selection 적용
        X_train_sel, X_val_sel, sel_cols = apply_feature_selection(X_train_imp, y_train, X_val_imp, n_features=30)
        selected_features_list.extend(sel_cols)

        # 3. 스케일링
        X_train_proc = scaler.fit_transform(X_train_sel)
        X_val_proc = scaler.transform(X_val_sel)

        # 4. SMOTE 적용 (오직 Train 세트에만 오버샘플링 적용)
        smote = SMOTE(random_state=42)
        X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)

        # 5. 모델 학습 및 확률 예측
        model.fit(X_train_res, y_train_res)

        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_val_proc)[:, 1]
        else:
            y_proba = model.predict(X_val_proc).astype(float)

        # 임계값(Threshold) 조정을 통한 최종 0/1 분류
        y_pred_custom = (y_proba >= CUSTOM_THRESHOLD).astype(int)

        all_y_true.extend(y_val)
        all_y_proba.extend(y_proba)

        # 지표 산출
        precision_vals, recall_vals, _ = precision_recall_curve(y_val, y_proba)
        pr_auc = auc(recall_vals, precision_vals)

        metrics['Accuracy'].append(accuracy_score(y_val, y_pred_custom))
        metrics['Precision'].append(precision_score(y_val, y_pred_custom, zero_division=0))
        metrics['Recall'].append(recall_score(y_val, y_pred_custom, zero_division=0))
        metrics['F1'].append(f1_score(y_val, y_pred_custom, zero_division=0))
        metrics['PR-AUC'].append(pr_auc)
        metrics['ROC-AUC'].append(roc_auc_score(y_val, y_proba))

    # 결과 저장
    comparison_results.append({
        'Model': model_name,
        'Accuracy': np.mean(metrics['Accuracy']),
        'Precision': np.mean(metrics['Precision']),
        'Recall (FN방지)': np.mean(metrics['Recall']),
        'F1-score': np.mean(metrics['F1']),
        'PR-AUC': np.mean(metrics['PR-AUC']),
        'ROC-AUC': np.mean(metrics['ROC-AUC'])
    })

    prec, rec, _ = precision_recall_curve(all_y_true, all_y_proba)
    pr_curves_data[model_name] = (rec, prec, auc(rec, prec))

    # 빈도수 기준 중요 Feature 기록
    feature_importances_dict[model_name] = pd.Series(selected_features_list).value_counts()

# ==============================================================================
# 4. 성능 비교표 출력
# ==============================================================================
df_comparison = pd.DataFrame(comparison_results).set_index('Model')
print("\n" + "=" * 70)
print(f"  [3주차 고도화] Feature Selection + SMOTE + Threshold({CUSTOM_THRESHOLD}) 적용 결과")
print("=" * 70)
print(df_comparison.round(4))

# ==============================================================================
# 5. [3주차 Step 3] Top 10 Feature 도출 (B/D 파트 협업 전달용)
# ==============================================================================
# 5개 Fold에서 가장 빈번하게 선별된 Top Feature 상위 10개 추출
top_10_features = feature_importances_dict['Logistic Regression'].head(10)

print("\n" + "=" * 70)
print("  [Step 3: Cross-Part Collaboration] B/D 파트 전달용 Top 10 Feature")
print("=" * 70)
print(top_10_features)

# ==============================================================================
# 6. PR Curve 시각화
# ==============================================================================
plt.figure(figsize=(9, 6))
for model_name, (rec, prec, score) in pr_curves_data.items():
    plt.plot(rec, prec, label=f'{model_name} (PR-AUC = {score:.4f})')

baseline_rate = sum(y) / len(y)
plt.axhline(y=baseline_rate, color='r', linestyle='--', label=f'Baseline ({baseline_rate:.4f})')
plt.xlabel('Recall (불량 탐지율)')
plt.ylabel('Precision (불량 정밀도)')
plt.title('3rd Week PR Curve Comparison (FS + SMOTE Applied)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'secom_25%_중앙값대치_이상치처리.xlsx'